# Sprint 3 v2 — MoE Training (Fixed)

**Use this notebook after `nb_fix_cache_rebuild.ipynb` has finished and you have saved the output cache as a Kaggle dataset.**

## What is different from Sprint 3 v1
| Change | v1 (broken) | v2 (this notebook) |
|---|---|---|
| Phase 2 `gate_only` | `true` — experts frozen | `false` — gate + experts co-adapt |
| Phase 2 epochs | 5 | 12 |
| Phase 2 `lr_gate` | 1e-3 | 5e-4 |
| Phase 2 `lr_experts` | 1e-4 | 5e-5 |
| Phase 2 `load_balance_weight` | 0.01 | 0.05 |
| Dataset sampler | `shuffle=True` (biased toward NIH/TBX11K) | `WeightedRandomSampler` (equal dataset contribution) |
| Cache supervision | Whole-lung prior fallback + ringify | Fixed: empty masks for failed proposals |

## Datasets to attach on Kaggle
| Dataset slug | Mount path |
|---|---|
| `iahmedhabib/medsam-vit-b` | `/kaggle/input/datasets/iahmedhabib/medsam-vit-b/` |
| `mabdullahi454/tb-pipeline-checkpoints` | `/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/` |
| *Your saved cache from Notebook 1* | `/kaggle/input/<your-cache-dataset>/moe_cache_v2` |
| `iahmedhabib/montgomery` | `/kaggle/input/datasets/iahmedhabib/montgomery/montgomery` |
| `iahmedhabib/shehzhenn` | `/kaggle/input/datasets/iahmedhabib/shehzhenn/shenzhen` |
| `usmanshams/tbx-11` | `/kaggle/input/datasets/usmanshams/tbx-11/TBX11K` |

**GPU:** T4 × 1 (required)

**Estimated runtime:**
- Phase 1 (expert pretraining, 10 epochs): ~50 min
- Phase 2 (joint MoE, 12 epochs): ~65 min
- Phase 3 (boundary critic, 5 epochs): ~15 min
- **Total: ~2.5 hours**

In [ ]:
# ── Cell 1: Clone repo and install dependencies ──────────────────────────────
import os, subprocess, sys
from pathlib import Path

REPO_URL  = "https://github.com/AhmedHabib00/dl-project-codebase.git"
REPO_DIR  = "/kaggle/working/repo"

# ---- UPDATE THIS PATH to wherever you saved the output of nb_fix_cache_rebuild.ipynb ----
# It should be a directory containing *.pt files named like montgomery__xxx.pt
CACHE_DIR = "/kaggle/input/<your-cache-dataset>/moe_cache_v2"  # <-- UPDATE THIS

SAVE_DIR  = "/kaggle/working/moe_v2"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     os.path.join(REPO_DIR, "requirements.txt")],
    check=True
)
print("Setup complete.")

In [ ]:
# ── Cell 2: Verify mounts and patch configs ──────────────────────────────────
import yaml
from pathlib import Path

MEDSAM_CKPT = Path("/kaggle/input/datasets/iahmedhabib/medsam-vit-b/medsam_vit_b.pth")
C1_ADAPTER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component1_adapters.safetensors")
C4_DECODER  = Path("/kaggle/input/datasets/mabdullahi454/tb-pipeline-checkpoints/component4_mask_decoder.pt")
CACHE_PATH  = Path(CACHE_DIR)

def check(label, path, required=True):
    status = "OK" if path.exists() else "MISSING"
    flag   = "[REQUIRED]" if required else "[optional]"
    print(f"  {flag} {label:<40}: {status}")
    if required and not path.exists():
        raise FileNotFoundError(f"{label} not found at {path}")

check("MedSAM checkpoint",     MEDSAM_CKPT)
check("C1 LoRA adapter",       C1_ADAPTER)
check("C4 lung decoder",       C4_DECODER)
check("MoE cache directory",   CACHE_PATH)

n_cache = len(list(CACHE_PATH.glob("*.pt")))
print(f"  Cache files: {n_cache}")
assert n_cache > 100, f"Cache is too small ({n_cache} files) — rebuild with nb_fix_cache_rebuild.ipynb first."

# Patch moe.yaml
moe_yaml = Path(REPO_DIR) / "configs" / "moe.yaml"
with moe_yaml.open() as f:
    cfg = yaml.safe_load(f)

cfg["component1"]["checkpoint_path"]          = str(MEDSAM_CKPT)
cfg["component1"]["adapter_path"]              = str(C1_ADAPTER)
cfg["component4"]["checkpoint_path"]          = str(MEDSAM_CKPT)
cfg["component4"]["decoder_checkpoint_path"]  = str(C4_DECODER)
cfg["moe_training"]["save_dir"]               = SAVE_DIR
# Ensure v2 settings are active (guard against stale YAML)
cfg["moe_training"]["joint"]["gate_only"]             = False
cfg["moe_training"]["joint"]["epochs"]                = 12
cfg["moe_training"]["joint"]["lr_gate"]               = 5e-4
cfg["moe_training"]["joint"]["lr_experts"]            = 5e-5
cfg["moe_training"]["joint"]["load_balance_weight"]   = 0.05
cfg["moe_training"]["joint"]["expert_aux_weight"]     = 0.5
cfg["moe_training"]["joint"]["resume_from_moe_best"]  = None

with moe_yaml.open("w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("\nConfig patched. Key Phase 2 settings:")
jcfg = cfg["moe_training"]["joint"]
for k in ("gate_only", "epochs", "lr_gate", "lr_experts", "load_balance_weight", "expert_aux_weight"):
    print(f"  {k}: {jcfg[k]}")

In [ ]:
# ── Cell 3: Pre-training sanity check — expert mask fractions in the cache ───
# Run this BEFORE training. If cavity is < 2% or NIH is > 10%, the cache was
# built with the old (broken) script. Rebuild with nb_fix_cache_rebuild.ipynb.

import torch
from collections import Counter
from pathlib import Path

expert_names = ["consolidation", "cavity", "fibrosis", "nodule"]
total_per_ds = Counter()
nonzero      = {e: Counter() for e in expert_names}

for p in Path(CACHE_DIR).glob("*.pt"):
    ds = p.stem.split("__")[0]
    total_per_ds[ds] += 1
    d = torch.load(p, weights_only=False)
    for e in expert_names:
        if float(d["expert_masks"][e].sum().item()) > 0:
            nonzero[e][ds] += 1

print("Dataset balance:")
for ds, cnt in sorted(total_per_ds.items()):
    print(f"  {ds:<20}: {cnt}")

total_all = sum(total_per_ds.values())
print("\nExpert non-zero fractions (all datasets):")
GATE_PASS = True
for e in expert_names:
    n = sum(nonzero[e].values())
    pct = 100.0 * n / total_all if total_all else 0.0
    ok = pct > 2.0
    if not ok:
        GATE_PASS = False
    print(f"  [{'OK  ' if ok else 'FAIL'}] {e:<16}: {pct:.1f}%")

nih_total = total_per_ds.get("nih_cxr14", 0)
if nih_total > 0:
    print("\nNIH expert non-zero fractions (should all be < 10%):")
    for e in expert_names:
        n = nonzero[e].get("nih_cxr14", 0)
        pct = 100.0 * n / nih_total
        ok = pct < 10.0
        if not ok:
            GATE_PASS = False
        print(f"  [{'OK  ' if ok else 'FAIL'}] {e:<16}: {pct:.1f}%")

if not GATE_PASS:
    raise RuntimeError(
        "Cache sanity check FAILED. Rebuild the cache with nb_fix_cache_rebuild.ipynb "
        "(the old cache from Sprint 3 v1 will not produce good results)."
    )
print("\nCache passed sanity check. Proceeding to training.")

In [ ]:
# ── Cell 4: Phase 1 — Expert pretraining (10 epochs, ~50 min) ───────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_experts",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
        "--all",
    ],
    cwd=REPO_DIR,
)
print(f"Phase 1 finished in {(time.time()-t0)/60:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 1 failed — check output above."

# train_experts.py saves <expert>_best.pt AND <expert>_final.pt per expert
best_ckpts = sorted(Path(SAVE_DIR).glob("expert_*_best.pt"))
print(f"Best expert checkpoints saved ({len(best_ckpts)}):")
for p in best_ckpts:
    print(f"  {p.name}")
assert len(best_ckpts) == 4, f"Expected 4 *_best.pt files, found {len(best_ckpts)}"


In [ ]:
# ── Cell 5: Phase 1 quick-eval — per-expert final loss should be < 0.45 ─────
# train_experts.py writes one history file per expert: expert_<name>_history.jsonl
import json
from pathlib import Path

EXPERTS = ["consolidation", "cavity", "fibrosis", "nodule"]
print("Expert final train losses (last epoch):")
for name in EXPERTS:
    log_file = Path(SAVE_DIR) / f"expert_{name}_history.jsonl"
    if not log_file.exists():
        print(f"  [WARN] {name:<16}: {log_file.name} not found")
        continue
    losses = []
    with log_file.open() as f:
        for line in f:
            try:
                e = json.loads(line)
                if "loss" in e:
                    losses.append(float(e["loss"]))
            except Exception:
                pass
    if not losses:
        print(f"  [WARN] {name:<16}: no loss entries in log")
        continue
    final = losses[-1]
    ok = final < 0.50
    print(f"  [{'OK  ' if ok else 'WARN'}] {name:<16}: {final:.4f}  ({len(losses)} epochs)")


In [ ]:
# ── Cell 6: Phase 2 — Joint MoE training, gate_only=False (12 epochs, ~65 min)
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_moe_joint",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
print(f"Phase 2 finished in {(time.time()-t0)/60:.1f} min  (return code: {result.returncode})")
assert result.returncode == 0, "Phase 2 failed — check output above."

moe_best = Path(SAVE_DIR) / "moe_best.pt"
assert moe_best.exists(), "moe_best.pt not found after Phase 2!"
print(f"moe_best.pt saved: {moe_best.stat().st_size / 1e6:.1f} MB")

In [ ]:
# ── Cell 7: Phase 2 convergence check ────────────────────────────────────────
# Healthy: mask_loss drops from ~0.50 to < 0.40 across 12 epochs.
# Healthy: lb_loss drops toward 1.0 (gate is specialising, not uniform).
import json
from pathlib import Path

log_file = Path(SAVE_DIR) / "moe_joint_history.jsonl"
if log_file.exists():
    epochs, mask_losses, lb_losses, totals = [], [], [], []
    with log_file.open() as f:
        for line in f:
            e = json.loads(line)
            epochs.append(e.get("epoch"))
            mask_losses.append(e.get("mask_loss", float("nan")))
            lb_losses.append(e.get("lb_loss", float("nan")))
            totals.append(e.get("total", float("nan")))

    print("Phase 2 training log:")
    print(f"  {'Epoch':<8} {'total':<10} {'mask_loss':<12} {'lb_loss':<10}")
    for ep, tot, ml, lb in zip(epochs, totals, mask_losses, lb_losses):
        print(f"  {ep:<8} {tot:<10.4f} {ml:<12.4f} {lb:<10.4f}")

    if len(mask_losses) >= 2:
        delta = mask_losses[0] - mask_losses[-1]
        print(f"\nMask loss drop: {delta:.4f}  ({'OK — model is learning' if delta > 0.03 else 'WARN — barely moved'})")
        lb_drop = lb_losses[0] - lb_losses[-1]
        print(f"LB loss drop:   {lb_drop:.4f}  ({'OK — routing specialising' if lb_drop > 0.05 else 'WARN — routing uniform'})")
else:
    print(f"Log file not found at {log_file} — training may still have succeeded.")


In [ ]:
# ── Cell 8: Phase 3 — Boundary critic (5 epochs, ~15 min) ───────────────────
import subprocess, sys, time
from pathlib import Path

t0 = time.time()
result = subprocess.run(
    [
        sys.executable, "-m", "src.training.train_boundary_critic",
        "--config",    str(Path(REPO_DIR) / "configs" / "moe.yaml"),
        "--cache-dir", CACHE_DIR,
    ],
    cwd=REPO_DIR,
)
print(f"Phase 3 finished in {(time.time()-t0)/60:.1f} min  (return code: {result.returncode})")
if result.returncode != 0:
    print("Phase 3 failed — boundary critic is optional; proceeding without it.")

In [ ]:
# ── Cell 9: Summarise outputs and quick inference sanity check ───────────────
import os
from pathlib import Path

print("Output artifacts:")
for p in sorted(Path(SAVE_DIR).glob("*.pt")):
    print(f"  {p.name:<50}: {p.stat().st_size/1e6:.1f} MB")

print()
print("Next step: run nb_eval_moe_v2.ipynb with these checkpoints.")
print(f"Point it at SAVE_DIR = '{SAVE_DIR}'")